# AI Intrusion Detection System — Data Exploration and Preprocessing

## 1. Objective

The objective of this notebook is to explore, clean, and prepare the NSL-KDD dataset for machine learning.

NSL-KDD contains records describing network connections. Each record includes 41 network traffic features, an attack label, and a difficulty score.


We will perform the following tasks:

1. Load the training and testing datasets.
2. Inspect the dataset structure, labels, missing values, and duplicates.
3. Analyze the distribution of normal and malicious connections.
4. Clean inconsistent values and prepare target labels.
5. Separate input features from target variables.
6. Encode categorical features for machine learning.
7. Save cleaned data in `datasets/processed/`.

## 2. Dataset strategy

The original training and testing files will remain unchanged in `datasets/raw/`. Any cleaned versions will be saved in `datasets/processed/`.

The initial prediction task is binary classification:

- `normal`: legitimate traffic according to the dataset labels.
- `attack`: any record whose original label represents an attack.

The dataset's difficulty score will be retained for reference but excluded from the model's input features.

## 3. Import the libraries
We use:
      pandas - to load and inspect tabular data.
      numpy  - to create the binary target.
      path - to work with project file paths
      

In [1]:
import pandas as pd 
import numpy as np 
from pathlib import Path

## 4. Define the dataset columns
The NSL-KDD files we downloaded don't include a header row. We supply the official 41 features names, followed by label and difficulty giving 43 columns in total.
The difficulty column is dataset metadata. We keep it for reference, but it will not be used as an input feature.

In [3]:
feature_names = [
    "duration", "protocol_type", "service", "flag",
    "src_bytes", "dst_bytes", "land", "wrong_fragment",
    "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted",
    "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count",
    "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate"
]

columns = feature_names + ["label", "difficulty"]

print("Number of network features:", len(feature_names))
print("Total columns including label and difficulty:", len(columns))

Number of network features: 41
Total columns including label and difficulty: 43


## 5. Load the training and testing datasets
We keep the official training and testing files separate:
     KDDTrain+.txt  is used to prepare the training data.
     KDDTest+.txt   is held separately for later model evaluation.
Using header=None tells pandas that the files don't contain column names.     

In [4]:
raw_dir = Path("../datasets/raw")

train_data = pd.read_csv(
    raw_dir / "KDDTrain+.txt",
    header=None,
    names=columns
) 
 
test_data = pd.read_csv(
    raw_dir / "KDDTest+.txt",
    header=None,
    names=columns
)

print("Training shape:", train_data.shape)
print("Testing shape:", test_data.shape)

Training shape: (125973, 43)
Testing shape: (22544, 43)


## 6. Preview and inspect the data
We inspect a few rows, data types, and numerical summaries to understand what was loaded.
head() shows the first 5 records.
info() shows columns data types and non-missing counts.
describe() sumarizes numerical coulumns.


In [6]:
display(train_data.head())
print("\nTraining Dataset Information:")
train_data.info()

print("\nSummary of numerical columns:")
display(train_data.describe())

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21



Training Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Data columns (total 43 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   duration                     125973 non-null  int64  
 1   protocol_type                125973 non-null  str    
 2   service                      125973 non-null  str    
 3   flag                         125973 non-null  str    
 4   src_bytes                    125973 non-null  int64  
 5   dst_bytes                    125973 non-null  int64  
 6   land                         125973 non-null  int64  
 7   wrong_fragment               125973 non-null  int64  
 8   urgent                       125973 non-null  int64  
 9   hot                          125973 non-null  int64  
 10  num_failed_logins            125973 non-null  int64  
 11  logged_in                    125973 non-null  int64  
 12  num_compromised              125973 no

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
count,125973.00000,1.259730e+05,1.259730e+05,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,...,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000
mean,287.14465,4.556674e+04,1.977911e+04,0.000198,0.022687,0.000111,0.204409,0.001222,0.395736,0.279250,...,115.653005,0.521242,0.082951,0.148379,0.032542,0.284452,0.278485,0.118832,0.120240,19.504060
std,2604.51531,5.870331e+06,4.021269e+06,0.014086,0.253530,0.014366,2.149968,0.045239,0.489010,23.942042,...,110.702741,0.448949,0.188922,0.308997,0.112564,0.444784,0.445669,0.306557,0.319459,2.291503
min,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,10.000000,0.050000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,18.000000
50%,0.00000,4.400000e+01,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,63.000000,0.510000,0.020000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,20.000000
75%,0.00000,2.760000e+02,5.160000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,255.000000,1.000000,0.070000,0.060000,0.020000,1.000000,1.000000,0.000000,0.000000,21.000000
max,42908.00000,1.379964e+09,1.309937e+09,1.000000,3.000000,3.000000,77.000000,5.000000,1.000000,7479.000000,...,255.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,21.000000


## 7. Check data quality
We count missing values and exact duplicate rows in each dataset. We do not automatically delete duplicates: this audit tells us whether there is an issue to investigate, while preserving the official datasets splits.

In [7]:
#Missing values in training and testing data.
print("Missing values in training data:", train_data.isnull().sum().sum())
print("Missing values in testing data:",test_data.isnull().sum().sum())

# Duplicate rows in training and testing data.
print("Duplicate rows in training data:", train_data.duplicated().sum())
print("Duplicate rows in testing data:", test_data.duplicated().sum())



Missing values in training data: 0
Missing values in testing data: 0
Duplicate rows in training data: 0
Duplicate rows in testing data: 0


## 8. Explore the original labels
The dataset contains several original labels: normal and names for diffrent attack types. We inspect the label distribution before simplifying it.
The training and testing files contain different attack categories. The distinction is useful later when interpreting how well a model generalizes.

In [8]:
print("Original training label counts:")
display(train_data["label"].value_counts())

print("\nOriginal testing label counts:")
display(test_data["label"].value_counts())

print("\nTraining label percentages:")
display(train_data["label"].value_counts(normalize=True).mul(100).round(2))

print("\nTesting label percentages:")
display(test_data["label"].value_counts(normalize=True).mul(100).round(2))

Original training label counts:


label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64


Original testing label counts:


label
normal             9711
neptune            4657
guess_passwd       1231
mscan               996
warezmaster         944
apache2             737
satan               735
processtable        685
smurf               665
back                359
snmpguess           331
saint               319
mailbomb            293
snmpgetattack       178
portsweep           157
ipsweep             141
httptunnel          133
nmap                 73
pod                  41
buffer_overflow      20
multihop             18
named                17
ps                   15
sendmail             14
xterm                13
rootkit              13
teardrop             12
xlock                 9
land                  7
xsnoop                4
ftp_write             3
loadmodule            2
worm                  2
perl                  2
sqlattack             2
udpstorm              2
phf                   2
imap                  1
Name: count, dtype: int64


Training label percentages:


label
normal             53.46
neptune            32.72
satan               2.88
ipsweep             2.86
portsweep           2.33
smurf               2.10
nmap                1.19
back                0.76
teardrop            0.71
warezclient         0.71
pod                 0.16
guess_passwd        0.04
buffer_overflow     0.02
warezmaster         0.02
land                0.01
imap                0.01
rootkit             0.01
loadmodule          0.01
ftp_write           0.01
multihop            0.01
phf                 0.00
perl                0.00
spy                 0.00
Name: proportion, dtype: float64


Testing label percentages:


label
normal             43.08
neptune            20.66
guess_passwd        5.46
mscan               4.42
warezmaster         4.19
apache2             3.27
satan               3.26
processtable        3.04
smurf               2.95
back                1.59
snmpguess           1.47
saint               1.42
mailbomb            1.30
snmpgetattack       0.79
portsweep           0.70
ipsweep             0.63
httptunnel          0.59
nmap                0.32
pod                 0.18
buffer_overflow     0.09
multihop            0.08
named               0.08
ps                  0.07
sendmail            0.06
xterm               0.06
rootkit             0.06
teardrop            0.05
xlock               0.04
land                0.03
xsnoop              0.02
ftp_write           0.01
loadmodule          0.01
worm                0.01
perl                0.01
sqlattack           0.01
udpstorm            0.01
phf                 0.01
imap                0.00
Name: proportion, dtype: float64

## 9. Create the binary target 
We preserve the original label column and create a separate target column:
   0 - when the original label is normal
   1-  for every other label representing an attack.
This changes the prediction task from identify a specific attack type to distingushing normal traffic from attack traffic.

In [9]:
# Convert the original labels into two classes
train_data["target"] = np.where(
    train_data["label"] == "normal", 0, 1
)

test_data["target"] = np.where(
    test_data["label"] == "normal", 0, 1
)

# Display the new class distribution
print("Training target distribution:")
print(train_data["target"].value_counts().sort_index())

print("\nTesting target distribution:")
print(test_data["target"].value_counts().sort_index())


Training target distribution:
target
0    67343
1    58630
Name: count, dtype: int64

Testing target distribution:
target
0     9711
1    12833
Name: count, dtype: int64


## 10. Identify Input  Features and Target Variables
The model inputs are the 41 network features. We excude:
     label - because it contains the original text label
     difficulty - because it is dataset metadata
     target -  because it is the answer the model will learn to predict.
We keep the official training and testing records separate.


In [10]:
X_train = train_data.drop(columns=["label", "difficulty", "target"])
y_train = train_data["target"]

X_test = test_data.drop(columns=["label", "difficulty", "target"])
y_test = test_data["target"]

print("Training features:", X_train.shape)
print("Training target:", y_train.shape)
print("Testing features:", X_test.shape)
print("Testing target:", y_test.shape)


Training features: (125973, 41)
Training target: (125973,)
Testing features: (22544, 41)
Testing target: (22544,)


## 11. Identifying Categorical and Numerical Features

The input dataset contains both categorical and numerical features.

Categorical features represent groups or types, such as network protocols, services, and connection flags. These features contain text and must be encoded into numerical representations before they can be used by our machine learning model.

We will identify the categorical columns and numerical columns separately. Later, we will apply one-hot encoding to the categorical features, fitting the encoder on the training data only to prevent information leakage from the test dataset.

In [11]:

# Identify categorical features
categorical_features = ["protocol_type", "service", "flag"]

# Identify numerical features
numerical_features = [
    column for column in X_train.columns
    if column not in categorical_features
]

# Display the feature groups
print("Categorical features:")
print(categorical_features)

print("\nNumber of categorical features:", len(categorical_features))

print("\nNumber of numerical features:", len(numerical_features))
print(numerical_features)

# Inspect the unique values in each categorical feature
for column in categorical_features:
    print(f"\nUnique values in {column}:")
    print(X_train[column].nunique())
    print(X_train[column].value_counts().head(10))


Categorical features:
['protocol_type', 'service', 'flag']

Number of categorical features: 3

Number of numerical features: 38
['duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']

Unique values in protocol_type:
3
protocol_type
tcp     102689
udp      14993
icmp      8291
Name: count, dtype: int64

Unique values in service:
70
service
http        40338
pri

## 12. Encoding Categorical Features

Machine learning models require numerical input. We will use one-hot encoding to convert the categorical features into binary indicator columns.

We will use a `ColumnTransformer` to apply one-hot encoding to `protocol_type`, `service`, and `flag`, while passing the 38 numerical features through unchanged.

The encoder will be fitted on the training data and reused to transform the testing data. Setting `handle_unknown="ignore"` ensures that categories appearing only in the test dataset do not cause an encoding error.

This approach keeps preprocessing consistent and prevents data leakage.

In [12]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Create the preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

# Learn the encoding from training data and transform it
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same encoding to testing data
X_test_processed = preprocessor.transform(X_test)

# Inspect the processed datasets
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)


Processed training shape: (125973, 122)
Processed testing shape: (22544, 122)


## 13. Validation of Preprocessed Data

These checks confirm that:
     The number of feature rows still matches the target rows.
     Training and testing have the same number of encoded columns.
     Encoded values contain no missing or infinite numbers.
     Both targets contain only 0 and 1.
     

In [13]:
from scipy import sparse

# Check that the row counts are unchanged
print("Training rows preserved:",
      X_train_processed.shape[0] == len(y_train))

print("Testing rows preserved:",
      X_test_processed.shape[0] == len(y_test))

# Check that training and testing feature dimensions match
print("Feature dimensions match:",
      X_train_processed.shape[1] == X_test_processed.shape[1])

# Check for missing and infinite values
if sparse.issparse(X_train_processed):
    train_values = X_train_processed.data
else:
    train_values = np.asarray(X_train_processed)

if sparse.issparse(X_test_processed):
    test_values = X_test_processed.data
else:
    test_values = np.asarray(X_test_processed)

print("Training data has no NaN values:",
      not np.isnan(train_values).any())

print("Testing data has no NaN values:",
      not np.isnan(test_values).any())

print("Training data has no infinite values:",
      np.isfinite(train_values).all())

print("Testing data has no infinite values:",
      np.isfinite(test_values).all())

# Confirm that the targets contain only 0 and 1
print("Training target classes:", sorted(y_train.unique()))
print("Testing target classes:", sorted(y_test.unique()))


Training rows preserved: True
Testing rows preserved: True
Feature dimensions match: True
Training data has no NaN values: True
Testing data has no NaN values: True
Training data has no infinite values: True
Testing data has no infinite values: True
Training target classes: [np.int64(0), np.int64(1)]
Testing target classes: [np.int64(0), np.int64(1)]


## 14. Save the prepared datasets and encoder
We save:
    Cleaned training and testing tables, including original labels and the binary target. 
    Encoded feature matcrices as compressed sparse files( .npz).
    Training and testing targets as CSV files.
    The fitted preprocessing transformer as a joblib file.

Keeping the cleaned tables means later experiments can create a validation split from the original training records and fit a fresh encoder on only the training portion of that split. The raw files rmain unchanged.
These file are generated when this cell is run; they are not stored in the raw-data folder.





In [21]:

from pathlib import Path
import joblib
from scipy import sparse

# Find the project root whether the notebook runs from
# the project folder or the notebooks folder.
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

# Create the processed directory if it does not exist.
processed_dir = project_root / "datasets" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Save cleaned tables with original labels and binary targets.
train_data.to_csv(processed_dir / "train_clean.csv", index=False)
test_data.to_csv(processed_dir / "test_clean.csv", index=False)

# Convert encoded features to sparse matrices and save them.
X_train_to_save = (
    X_train_processed.tocsr()
    if sparse.issparse(X_train_processed)
    else sparse.csr_matrix(X_train_processed)
)

X_test_to_save = (
    X_test_processed.tocsr()
    if sparse.issparse(X_test_processed)
    else sparse.csr_matrix(X_test_processed)
)

sparse.save_npz(
    processed_dir / "X_train_encoded.npz",
    X_train_to_save
)
sparse.save_npz(
    processed_dir / "X_test_encoded.npz",
    X_test_to_save
)

# Save the training and testing targets.
y_train.to_frame(name="target").to_csv(
    processed_dir / "y_train.csv", index=False
)
y_test.to_frame(name="target").to_csv(
    processed_dir / "y_test.csv", index=False
)

# Save the fitted preprocessing transformer.
joblib.dump(
    preprocessor,
    processed_dir / "preprocessor.joblib"
)

# Confirm which files were saved.
print(f"Saved prepared files to: {processed_dir.resolve()}")

for path in sorted(processed_dir.iterdir()):
    if path.is_file():
        print(f"- {path.name} ({path.stat().st_size:,} bytes)")


Saved prepared files to: C:\Users\NIPSEY\Desktop\AI-Intrusion-Detection-System\datasets\processed
- .gitkeep (0 bytes)
- preprocessor.joblib (5,329 bytes)
- test_clean.csv (3,239,276 bytes)
- train_clean.csv (17,973,453 bytes)
- X_test_encoded.npz (444,520 bytes)
- X_train_encoded.npz (2,465,795 bytes)
- y_test.csv (67,640 bytes)
- y_train.csv (377,927 bytes)


## 15. Final Summary
This notebook has explored the NSL-KDD dataset, created the binary target, identified categorical and numerical features, encoded the categorical inputs, validated the results , and saved the prepared files.
 
 #### Key Decisions
    The raw training and testing files were not modified. 
    The original label and difficulty columns were excluded from model inputs.
    The binary target is 0 or normal traffic and 1 for attack traffic.
    The encoder was fitted on training features and reused for test features.
    No model  training or model-performance comparisons are performed in this notebook.


Next step: use the saved cleaned tables in the model-training workflow. If performing a train/validation experiment, split the training table first and fit a new encoder only on the training portion. Keep the offical test set for final evaluation.    